# Working with Text Data

In [2]:
import os

In [3]:
with open("the-verdict.txt", "r") as file:
    content = file.read()

print("Total characters:", len(content))    
print(content[:100])

Total characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [4]:
import re

text="Hello, world! Welcome to LLM training from scratch."
result=re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world!', ' ', 'Welcome', ' ', 'to', ' ', 'LLM', ' ', 'training', ' ', 'from', ' ', 'scratch.']


In [5]:
result=re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world!', ' ', 'Welcome', ' ', 'to', ' ', 'LLM', ' ', 'training', ' ', 'from', ' ', 'scratch', '.', '']


Let us try more complex regular expression handling other types of punctuation 

In [6]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [7]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', content)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:100])


['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']


# Converting tokens to token IDs

In [8]:
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1130


In [9]:
vocab={word:idx for idx, word in enumerate(all_words)}
# vocab

Building a tokenizer class to do all things explained till now 

In [10]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [11]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [12]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## Adding special context tokens

We add special tokens for unknown words and to specify end of a text

In [19]:
all_tokens= sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])

vocab={word:idx for idx, word in enumerate(all_tokens)}

Notice that the length of vocab increased by 2

In [15]:
len(vocab)

1132

Let us add `<unk>` token to the tokenizer

In [20]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Below is an example use of V2 tokenizer

In [21]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, would you like something?"
text2 = "Yes please, in the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, would you like something? <|endoftext|> Yes please, in the sunlit terraces of the palace.


In [22]:
tokenizer.encode(text)

[1131,
 5,
 1120,
 1126,
 628,
 912,
 10,
 1130,
 112,
 1131,
 5,
 568,
 988,
 956,
 984,
 722,
 988,
 1131,
 7]

In [23]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, would you like something? <|endoftext|> Yes <|unk|>, in the sunlit terraces of the <|unk|>.'

## Tokenizer - Byte Pair Encoding

In [24]:
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [tiktoken]5/7 [requests]ormalizer]


In [25]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [26]:
tokenizer=tiktoken.get_encoding("gpt2")

In [33]:
text = ("Hello, would you like something? <|endoftext|> Yes please, in the sunlit terraces of the palace."
)

integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)
strings=tokenizer.decode(integers)
print(strings)

[15496, 11, 561, 345, 588, 1223, 30, 220, 50256, 3363, 3387, 11, 287, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]
Hello, would you like something? <|endoftext|> Yes please, in the sunlit terraces of the palace.
